# CCNY Fall 2021 Calendar Scraper

Scrape the academic calendar using requests, Beautiful Soup, and pandas.

In [2]:
%pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

url = "https://www.ccny.cuny.edu/registrar/fall"
fallback = Path("data/fall_2021.html")

try:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    html = response.text
    source = "Live CCNY page (requests)"
except requests.RequestException as error:
    if not fallback.exists():
        raise RuntimeError("Download failed and data/fall_2021.html is missing.") from error
    html = fallback.read_text(encoding="utf-8")
    source = "Saved browser calendar table (live request failed)"
    print(f"Live download unavailable: {type(error).__name__}")

soup = BeautifulSoup(html, "html.parser")
if not soup.find("table") or "Fall 2021 Academic Calendar" not in soup.get_text():
    raise ValueError("The response does not contain the expected calendar.")

print("Source:", source)
print(soup.title.get_text(strip=True))


Live download unavailable: HTTPError
Source: Saved browser calendar table (live request failed)
Fall 2021 Academic Calendar


### Download issue

I got a 403 Forbidden error when I tried downloading the calendar with requests. Adding a User-Agent header did not fix it. The notebook tries the website first, then uses a local copy of the calendar table if the download fails. This lets me continue parsing the calendar with Beautiful Soup, but the live download with requests is still blocked.